# FarmPulse - MLOps tracking

Notebook de démonstration: DVC pour les données, MLflow pour les runs/artifacts, ONNX pour le déploiement, Docker et CI pour la reproductibilité.

In [ ]:
from pathlib import Path
import sys, json, subprocess, mlflow

ROOT = Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(subprocess.run(['dvc', 'status'], cwd=ROOT, capture_output=True, text=True).stdout)
print((ROOT / 'params.yaml').read_text())

In [ ]:
from src.utils.tracking import mlflow_tracking_uri
mlflow.set_tracking_uri(mlflow_tracking_uri(ROOT))
client = mlflow.MlflowClient()
experiments = client.search_experiments()
[(experiment.name, experiment.experiment_id) for experiment in experiments]

In [ ]:
runs = client.search_runs(experiment_ids=[e.experiment_id for e in experiments], max_results=20, order_by=['attribute.start_time DESC']) if experiments else []
[(run.info.run_id, run.info.run_name, run.data.metrics) for run in runs]

In [ ]:
checkpoint = ROOT / 'models' / 'best.pt'
if checkpoint.exists():
    from src.models.export_onnx import export
    export(checkpoint, ROOT / 'models' / 'model.onnx')
    print('ONNX exported')
else:
    print('Train first: models/best.pt does not exist yet')

## Commandes de livraison

```powershell
dvc repro
pytest backend/tests -q
docker compose up --build
```

Pour la soutenance, montrer l'historique MLflow, `dvc status`, le build CI et l'inférence Flutter.